# 🩺 DoCare AI - Multi-GPU Parallel Fine-Tuning (Kaggle 2x Tesla T4)
## Melatih LLM Medis Menggunakan Dataset Percakapan Dokter-Pasien Terverifikasi (Certified Doctor-Patient Consultations)

---

### ⚙️ Konfigurasi Akselerator (Kaggle Accelerator: 30 Jam/Minggu)
- **Pilihan Akselerator**: **GPU T4 x 2** (Dual NVIDIA Tesla T4 @ 16GB = **32 GB Total VRAM**, 30 GB System RAM).
- **Mengapa T4 x 2 lebih direkomendasikan dibanding TPU v5e-8 untuk task ini?**
  - **GPU T4 x 2**: Mendukung ekosistem standar PyTorch, CUDA, 4-bit Quantization (`bitsandbytes` NF4), LoRA (`peft`), dan **DDP (Distributed Data Parallelism)** natively.
  - **TPU v5e-8**: Sangat bertenaga namun tidak mendukung kernel CUDA (`bitsandbytes`), membutuhkan library PyTorch/XLA khusus, dan rawan incompatible dengan HuggingFace SFTTrainer quant.

---

### 📚 Sumber Data Medis Terverifikasi (Certified Sources)
Training ini menggunakan dataset percakapan klinis nyata antara pasien dan dokter spesialis berlisensi:
1. **`lavita/ChatDoctor-HealthCareMagic-100k`**: 100.000 dialog konsultasi nyata pasien dengan dokter bersertifikasi (HealthCareMagic Platform).
2. **`lavita/ChatDoctor-iCliniq`**: 10.000 konsultasi klinis dari dokter platform iCliniq.
3. **`MedQuAD`**: Basis pengetahuan tanya-jawab medis dari National Institutes of Health (NIH).

In [ ]:
# 1. Verifikasi Akselerator Dual GPU T4 (Kaggle 30 Jam/Minggu)
!nvidia-smi

In [ ]:
# 2. Instalasi Dependensi Terkini untuk Multi-GPU & LoRA
!pip install -q -U "transformers>=4.45.0" "datasets>=2.20.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" trl accelerate

In [ ]:
# 3. Pemeriksaan Dukungan Paralelisme PyTorch Multi-GPU
import torch
gpu_count = torch.cuda.device_count()
print(f"🔥 Total GPU Terdeteksi: {gpu_count}")
for i in range(gpu_count):
    print(f"   [GPU {i}] {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

if gpu_count < 2:
    print("⚠️ PERINGATAN: Hanya 1 GPU terdeteksi! Pastikan di panel kanan Settings -> Accelerator dipilih 'GPU T4 x2'.")
else:
    print("✅ Siap untuk Distributed Data Parallel (DDP) training di kedua GPU!")

In [ ]:
# 5. Susun Skrip Training Multi-GPU Paralel (train_docare_ddp.py)
# Skrip ini akan dieksekusi oleh torchrun di GPU 0 dan GPU 1 secara paralel!

script_content = '''import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Konfigurasi Multi-GPU DDP
local_rank = int(os.environ.get("LOCAL_RANK", 0))
world_size = int(os.environ.get("WORLD_SIZE", 1))
torch.cuda.set_device(local_rank)

print(f"[Worker Process {local_rank}/{world_size}] Menginisialisasi GPU: {torch.cuda.get_device_name(local_rank)}")

# 2. Model & Tokenizer
model_id = "unsloth/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": local_rank}
)
model = prepare_model_for_kbit_training(model)

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

# 4. Memuat & Memfilter Dataset Percakapan Dokter Bersertifikasi (Quality Control)
try:
    raw_dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train[:6000]")
except Exception:
    raw_dataset = load_dataset("lavita/ChatDoctor-iCliniq", split="train[:6000]")

def format_dialogue(batch):
    texts = []
    for inp, out in zip(batch.get("input", []), batch.get("output", [])):
        # Quality Control: Hanya sertakan keluhan dan saran medis yang lengkap
        if inp and out and len(inp.strip()) >= 15 and len(out.strip()) >= 20:
            patient_text = inp.strip()
            doctor_text = out.strip()
            formatted = (
                f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
                f"Anda adalah DoCare AI, asisten pendamping kesehatan terpercaya berbasis rujukan dokter bersertifikasi. "
                f"Jawab pertanyaan pasien dengan sopan, empatik, jelas, dan akurat secara medis.<|eot_id|>\n"
                f"<|start_header_id|>user<|end_header_id|>\n{patient_text}<|eot_id|>\n"
                f"<|start_header_id|>assistant<|end_header_id|>\n"
                f"Terima kasih telah berkonsultasi dengan DoCare AI. {doctor_text} "
                f"Semoga informasi ini bermanfaat dan lekas sembuh.<|eot_id|>"
            )
            texts.append(formatted)
    return {"text": texts}

dataset = raw_dataset.map(format_dialogue, batched=True)
dataset = dataset.filter(lambda x: len(x.get("text", "")) > 50)

# 5. Training Arguments Multi-GPU Paralel DDP
output_dir = "/kaggle/working/docare_llama_lora_ddp"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,          # Batch 2 per GPU (Effective batch 4 across 2 GPUs)
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=150,
    fp16=True,
    optim="paged_adamw_8bit",
    ddp_find_unused_parameters=False,       # Optimasi sinkronisasi gradient DDP
    save_strategy="steps",
    save_steps=75,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

if local_rank == 0:
    print("🚀 Memulai Training Paralel Multi-GPU (GPU 0 & GPU 1 aktif bersamaan)...")

trainer.train()

if local_rank == 0:
    print("💾 Menyimpan LoRA Adapter final...")
    model.save_pretrained(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    print("✅ Training Paralel Selesai!")
'''

with open('train_docare_ddp.py', 'w', encoding='utf-8') as f:
    f.write(script_content)

print('✅ Skrip train_docare_ddp.py berhasil dibuat dan siap untuk torchrun multi-GPU!')

In [ ]:
# 5. Susun Skrip Training Multi-GPU Paralel (train_docare_ddp.py)
# Skrip ini akan dijalankan dengan torchrun across BOTH GPU 0 and GPU 1 secara paralel!

script_content = '''
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Local rank untuk Multi-GPU DDP
local_rank = int(os.environ.get("LOCAL_RANK", 0))
world_size = int(os.environ.get("WORLD_SIZE", 1))
torch.cuda.set_device(local_rank)

print(f"[Process {local_rank}/{world_size}] Menginisialisasi GPU: {torch.cuda.get_device_name(local_rank)}")

# 1. Model & Tokenizer
model_id = "unsloth/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"" : local_rank}
)
model = prepare_model_for_kbit_training(model)

# 2. LoRA Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

# 3. Format Data Percakapan Medis Resmi
try:
    raw_dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train[:6000]")
except Exception:
    raw_dataset = load_dataset("lavita/ChatDoctor-iCliniq", split="train[:6000]")

def format_dialogue(batch):
    texts = []
    for inp, out in zip(batch.get("input", []), batch.get("output", [])):
        # Quality Control (QC): Filter dialog yang terlalu pendek atau null\n",
        if inp and out and len(inp.strip()) >= 15 and len(out.strip()) >= 20:\n",
            patient_text = inp.strip()\n",
            doctor_text = out.strip()\n",
        formatted = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n"
            f"Anda adalah DoCare AI, asisten pendamping kesehatan terpercaya berbasis saran dokter tersertifikasi. Jawab keluhan secara profesional, empatik, dan akurat.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\\n{patient_text}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\\nTerima kasih telah berkonsultasi dengan DoCare AI. {doctor_text} Semoga lekas sembuh.<|eot_id|>"
        )
        texts.append(formatted)
    return {"text": texts}

dataset = raw_dataset.map(format_dialogue, batched=True)
dataset = dataset.filter(lambda x: len(x.get("text", "")) > 50)\n",

# 4. Training Arguments dengan DDP Multi-GPU Paralel
output_dir = "/kaggle/working/docare_llama_lora_ddp"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,          # Batch 2 per GPU (Total effective batch: 4)
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=120,
    fp16=True,
    optim="paged_adamw_8bit",
    ddp_find_unused_parameters=False,       # Optimasi DDP Multi-GPU
    save_strategy="steps",
    save_steps=60,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

if local_rank == 0:
    print("🚀 Memulai Training Multi-GPU Paralel (GPU 0 & GPU 1)... ")

trainer.train()

if local_rank == 0:
    print("💾 Menyimpan Model LoRA Final...")
    model.save_pretrained(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    print("✅ Training Paralel Selesai 100%!")
'''

with open("train_docare_ddp.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("Skrip train_docare_ddp.py berhasil dibuat!")

In [ ]:
# 6. Jalankan Multi-GPU Parallel Training Menggunakan torchrun (2 GPU Processes)
# Ini akan mengeksekusi training pada GPU 0 dan GPU 1 secara paralel bersamaan!
!torchrun --nproc_per_node=2 train_docare_ddp.py

In [ ]:
# 7. Pengujian Model Hasil Training (Inference Test)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "unsloth/Llama-3.2-3B-Instruct"
lora_path = "/kaggle/working/docare_llama_lora_ddp/final"

print("Memuat model untuk pengujian inferensi...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, lora_path)

prompt = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
    "Anda adalah DoCare AI, asisten pendamping kesehatan terpercaya.<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n"
    "Dokter, saya mengalami pusing berputar, mual saat berdiri, dan telinga berdenging. Apa kira-kira penyebabnya?<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.3)

print("\n--- Respon DoCare AI (Hasil Trained Model) ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

In [ ]:
# 8. Kompres Hasil Model agar Siap Diunduh ke Laptop
!tar -czvf /kaggle/working/docare_llama_lora_ddp.tar.gz -C /kaggle/working/docare_llama_lora_ddp/final .
print("\n🎉 SELESAI! File /kaggle/working/docare_llama_lora_ddp.tar.gz siap diunduh dari tab Output Kaggle!")